## Code to Create Synthetic Test Dataset
- Use publicly available dataset for job descriptions from LinkedIn
- Use a powerful model to generate synthetic CV that can be used as an evaluation set

In [ ]:
# import rt libraries"""
import pandas as pd

import os



from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate

from langchain_anthropic import ChatAnthropic

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
df_jd = pd.read_csv('../datasets/clean_jobs.csv')

In [4]:
df_jd.head()

,id,title,company,location,link,source,date_posted,work_type,employment_type,description
0,1,Data Analyst,Meta,"New York, NY",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
1,2,Data Analyst,Meta,"San Francisco, CA",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
2,3,Data Analyst,Meta,"Los Angeles, CA",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
3,4,Data Analyst,Meta,"Washington, DC",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
4,5,Data Analyst II,Pinterest,"Chicago, IL",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-16,NaN,NaN,About Pinterest\n\nMillions of people around t...


In [ ]:
## Selected jobs - Select some job titles based on manual inspection of the records. 
sel_titles = ['Marketing Data Analyst','Senior Data Scientist', 'Data Engineer - Commerce Platform', 'AI Research Scientist', 'Senior C# Developer', 'Senior IT Business Analyst (MDA)' ]
# Show the 'description' field for matching title. 
df_selected_jd = df_jd[df_jd['title'].isin(sel_titles)].drop_duplicates('title')[['title', 'description']]

In [9]:
df_selected_jd.head()

,title,description
18,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...
122,Senior Data Scientist,Teamwork makes the stream work.\nRoku is chang...
313,Data Engineer - Commerce Platform,We are seeking a Data engineer to join our com...
441,AI Research Scientist,What you can expect\n\nAs a Research Scientist...
892,Senior IT Business Analyst (MDA),COLSA Corporation is seeking candidates for an...


In [10]:
df_selected_jd.shape

(6, 2)

In [ ]:
## Create LLM to generate some synthetic CVs for evaluation
# Use a powerful model to generate mostly accurate CVs
llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    max_tokens=4096,           
    temperature=0.7,            
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

In [12]:
# Create a prompt to generate multiple CV with varying degree of fit for each job
prompt_template = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an advanced synthetic data engine configured for HR tech evaluation. "
        "Your task is to generate a realistic, high-fidelity professional CV/Resume based on a "
        "Job Description and a strict Target Match Tier. Ignore information like company overview, job location and nature of contract in the job description.\n\n"
        "CRITICAL FIT CONSTRAINTS:\n"
        "- Strong Match (Perfect): Perfect alignment with target technical stack, seniority, and industry experience.\n"
        "- Strong Match (Semantic Variant): Candidate has full expertise but intentionally uses alternative terminology, "
        "framework synonyms, or parallel toolsets to thoroughly test semantic embedding coverage.\n"
        "- Average Match (Junior/Underqualified): Matches the required technical tools but lacks structural design leadership, "
        "scale, or the requested years of seniority.\n"
        "- Average Match (Tech Gaps): Correct industry level and background, but misses roughly half of the target stack.\n"
        "- Bad Match: Highly developed, senior-level profile but in an completely unrelated tech vertical or operational area "
        "(e.g., a specialized UX frontend developer or sales pipeline strategist applying for a deep platform data infrastructure / GenAI pipeline role).\n\n"
        "Format the result cleanly using standard resume markdown headers (Summary, Core Competencies, Professional History, Education). "
        "Provide raw resume text only. Do not wrap your response in conversational small talk or meta-explanations."
    )),
    ("human", "### TARGET MATCH TIER:\n{match_tier}\n\n### JOB TITLE:\n{title}\n\n### JOB DESCRIPTION:\n{description}")
])

In [13]:
# Create the chain 
generation_chain = prompt_template | llm | StrOutputParser()

In [14]:
match_tiers = [
    "Strong Match (Perfect alignment with core skills, technical stack, and target seniority)",
    "Strong Match (High alignment but leverages alternative framework synonyms and parallel engineering concepts)",
    "Average Match (Lacks the target seniority level or has a gap in senior architectural execution)",
    "Average Match (Solid industry context but missing half of the specific tools described)",
    "Bad Match (Excellent profile but in a completely unrelated engineering field or operational domain)"
]

In [24]:
# Save the generated CVs as a list of strings
generated_data = []
# test the generation with 1 sample
row = df_selected_jd.iloc[0]
jd_title  , jd_desc  = row['title'], row['description']
tier = match_tiers[0]
print(f"Generating profiles for Position: {jd_title}")
resume_output = generation_chain.invoke({
    "title": jd_title,
    "description": jd_desc,
    "match_tier": tier
})
generated_data.append( resume_output)

Generating profiles for Position: Marketing Data Analyst


In [25]:
resume_output

'# Jordan M. Calloway\n**Senior Marketing Data Analyst | BI Engineer**\nDallas, TX (CST) | jordan.calloway@email.com | (214) 555-0193 | linkedin.com/in/jordancalloway\n\n---\n\n## Summary\n\nResults-driven Senior Marketing Data Analyst and BI Engineer with 9+ years of experience delivering data-driven insights within financial services and banking environments. Proven track record designing and deploying enterprise-grade Tableau dashboards, authoring complex SQL logic, and translating ambiguous client requirements into actionable reporting solutions. Equally comfortable presenting findings to C-suite executives as architecting backend query logic. Recognized for outstanding client relationship management, cross-functional stakeholder communication, and consistently exceeding delivery expectations on high-visibility analytics engagements.\n\n---\n\n## Core Competencies\n\n- **Business Intelligence & Visualization:** Tableau Desktop, Tableau Server, Tableau Prep, DOMO, SSRS, Power BI\n- 

In [ ]:
# Loop through each job description and create synthetic CVs
for index, row in df_selected_jd.iterrows():
    
    jd_title = row['title'] 
    jd_desc = row['description']
    
    print(f"Generating profiles for Position [{index + 1}/6]: {jd_title}")
    
    for tier in match_tiers:
        try:
            # Generate the response via the LCEL execution pipeline
            resume_output = generation_chain.invoke({
                "title": jd_title,
                "description": jd_desc,
                "match_tier": tier
            })
            
            # Extract basic category for downstream analytical tagging
            clean_tier_label = tier.split(" (")[0]
            
            generated_data.append({
                "source_jd_title": jd_title,
                "source_jd_description": jd_desc,
                "assigned_tier": clean_tier_label,
                "tier_nuance_instruction": tier,
                "generated_resume_text": resume_output
            })
            
        except Exception as error:
            print(f"Skipping execution block for {jd_title} under condition '{tier}' due to: {error}")

Generating profiles for Position [19/6]: Marketing Data Analyst
Generating profiles for Position [123/6]: Senior Data Scientist
Generating profiles for Position [314/6]: Data Engineer - Commerce Platform
Generating profiles for Position [442/6]: AI Research Scientist
Generating profiles for Position [893/6]: Senior IT Business Analyst (MDA)
Generating profiles for Position [1008/6]: Senior C# Developer


In [21]:
df_controlled_eval_set = pd.DataFrame(generated_data)

print(f"\nProcessing complete! Evaluation dataset created with {len(df_controlled_eval_set)} total entries.")


Processing complete! Evaluation dataset created with 30 total entries.


In [22]:
df_controlled_eval_set.head()

,source_jd_title,source_jd_description,assigned_tier,tier_nuance_instruction,generated_resume_text
0,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Strong Match,Strong Match (Perfect alignment with core skil...,"# Jordan M. Calloway\nDallas, TX | jordan.call..."
1,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Strong Match,Strong Match (High alignment but leverages alt...,"# Jordan M. Calloway\nDallas, TX | (214) 555-0..."
2,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Average Match,Average Match (Lacks the target seniority leve...,# Jordan M. Calloway\n📧 jordan.calloway@email....
3,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Average Match,Average Match (Solid industry context but miss...,"# Jordan M. Calloway\n📍 Nashville, TN (CST) | ..."
4,Marketing Data Analyst,Title: Sr BI Engineer/ Sr Marketing Data Analy...,Bad Match,Bad Match (Excellent profile but in a complete...,"# Jordan M. Castellano\nAustin, TX | (512) 448..."


In [23]:
# save the dataset in ../datasets folder
df_controlled_eval_set.to_csv('../datasets/jd_resume_pairs.csv')